# TCN Training Only (Clean)

This notebook is for **training only**.
It uses isolated `train_*` variables and a Sharpe-based checkpoint policy.

## 1) Connect to Colab VM and Sync Repo
Run this first.

In [1]:
# Fresh-start cleanup cell (run before importing project modules)
import gc
import shutil
import subprocess
import sys
from pathlib import Path

TRAIN_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
TRAIN_REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")

# 1) Sync repo to latest main
if not (TRAIN_REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", TRAIN_REPO_URL, str(TRAIN_REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(TRAIN_REPO_DIR), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(TRAIN_REPO_DIR), "reset", "--hard", "origin/main"], check=True)

# 2) Remove old experiment outputs/checkpoints/cached data
purge_paths = [
    TRAIN_REPO_DIR / "tcn_fusion_results",
    TRAIN_REPO_DIR / "tcn_results",
    TRAIN_REPO_DIR / "tcn_att_results",
    TRAIN_REPO_DIR / "output_logs",
    TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
    TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
    TRAIN_REPO_DIR / "data" / "daily_ohlcv_assets.csv",              # forces fresh OHLCV download
    TRAIN_REPO_DIR / "data" / "processed_daily_macro_features.csv",   # forces fresh macro cache build
]

deleted = []
for p in purge_paths:
    if p.is_dir():
        shutil.rmtree(p, ignore_errors=True)
        deleted.append(str(p))
    elif p.is_file():
        p.unlink(missing_ok=True)
        deleted.append(str(p))

# 3) Remove Python/Jupyter cache folders
for cache_dir in TRAIN_REPO_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)
for ckpt_dir in TRAIN_REPO_DIR.rglob(".ipynb_checkpoints"):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

# 4) Clear loaded project modules from kernel memory
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod.startswith("src_"):
        del sys.modules[mod]
gc.collect()

print("✅ Fresh start complete")
print(f"Repo: {TRAIN_REPO_DIR}")
print(f"Deleted paths: {len(deleted)}")
for d in deleted:
    print(" -", d)

✅ Fresh start complete
Repo: /content/tcn_tape_vectorized_version_clean
Deleted paths: 0


In [2]:
#from pathlib import Path
import os

root = Path("/content/tcn_tape_vectorized_version_clean")
print("Exists:", root.exists())
print("CWD:", os.getcwd())

print("\nTop-level:")
for p in sorted(root.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f" - [{kind}] {p.name}")

# Quick check for outputs/caches you expected to be deleted
targets = [
    "tcn_fusion_results",
    "tcn_results",
    "tcn_att_results",
    "output_logs",
    "data/phase1_preparation_artifacts",
    "data/master_features_NORMALIZED.csv",
    "data/daily_ohlcv_assets.csv",
    "data/processed_daily_macro_features.csv",
]
print("\nTarget paths:")
for t in targets:
    p = root / t
    print(f" - {t}: {'EXISTS' if p.exists() else 'MISSING'}")


Exists: True
CWD: /content

Top-level:
 - [DIR ] .git
 - [FILE] .gitignore
 - [FILE] RL Portfolio Optimization Feature Engineering.md
 - [FILE] RL_Portfolio_Optimization_Feature_Engineering.ipynb
 - [FILE] USAGE_GUIDE_ACTUARIAL.py
 - [FILE] __init__.py
 - [FILE] convert_md_to_ipynb.py
 - [DIR ] data
 - [FILE] debug_attention_weights.py
 - [DIR ] docs
 - [DIR ] eval
 - [DIR ] paper
 - [DIR ] prompts
 - [FILE] ra_kl_research_writeup.ipynb
 - [FILE] rcdcc_research_writeup.ipynb
 - [FILE] requirements.txt
 - [FILE] run_tcn_eval.py
 - [DIR ] src
 - [FILE] tcn_architecture_analysis.ipynb
 - [DIR ] tcn_documentation
 - [FILE] tcn_evaluation_only.ipynb
 - [FILE] technical_deep_dive_presentation.ipynb
 - [DIR ] tests
 - [FILE] traditional_portfolio_benchmarks.ipynb
 - [DIR ] training_scripts

Target paths:
 - tcn_fusion_results: MISSING
 - tcn_results: MISSING
 - tcn_att_results: MISSING
 - output_logs: MISSING
 - data/phase1_preparation_artifacts: MISSING
 - data/master_features_NORMALIZED.csv

In [3]:
#!find /content/tcn_tape_vectorized_version_clean -maxdepth 3 | head -n 300

In [4]:
# Install project requirements in Colab VM
#import subprocess, sys
#from pathlib import Path

REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")
REQ_FILE = REPO_DIR / "requirements.txt"

if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

print("Using python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)], check=True)

print("✅ Requirements installed")


Using python: /usr/bin/python3
✅ Requirements installed


In [5]:
# --- GPU sanity/setup for TensorFlow ---
import tensorflow as tf

!nvidia-smi -L

gpus = tf.config.list_physical_devices("GPU")
print("TF GPUs:", gpus)
if not gpus:
    raise RuntimeError("No GPU visible to TensorFlow. In Colab: Runtime -> Change runtime type -> GPU")

for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

USE_MIXED_PRECISION = False  # set True only after stable fp32 training
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
else:
    tf.keras.mixed_precision.set_global_policy("float32")
print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())

with tf.device("/GPU:0"):
    a = tf.random.normal((4096, 4096))
    b = tf.random.normal((4096, 4096))
    c = tf.matmul(a, b)

print("Matmul device:", c.device)
print("Default GPU device name:", tf.test.gpu_device_name())

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-24e78449-37cd-bc81-9a3a-4bfe3b7001a4)
TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
Matmul device: /job:localhost/replica:0/task:0/device:GPU:0
Default GPU device name: /device:GPU:0


In [ ]:
import numpy, pandas, tensorflow
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("tensorflow", tensorflow.__version__)

## 2) Imports

In [6]:
import os, sys
from pathlib import Path

REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")

if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repo not found: {REPO_DIR}")

# Set working directory
os.chdir(REPO_DIR)

# Add repo root to Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

cwd: /content/tcn_tape_vectorized_version_clean
sys.path[0]: /content/tcn_tape_vectorized_version_clean


In [7]:
from copy import deepcopy
from pathlib import Path

import pandas as pd

from src.config import get_active_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import prepare_phase1_dataset, run_experiment6_tape

## 3) Base Config and Dataset Prep

In [8]:
# ------------------------------------------------------------------
# Global feature-audit plan enforcement (49 + 4 actuarial = 53)
# ------------------------------------------------------------------

def enforce_feature_audit_plan(cfg):
    fs = cfg.setdefault("feature_params", {}).setdefault("feature_selection", {})
    fs["enforce_allowlist"] = True
    fs["allowlist_apply_to_phase2"] = False

    allowlist = list(dict.fromkeys(fs.get("active_features_allowlist", []) or []))
    fs["active_features_allowlist"] = allowlist

    plan_name = fs.get("feature_audit_plan_name", "feature_audit_allowlist")
    expected_total = int(fs.get("feature_audit_expected_total_count", len(allowlist)))
    act_cols = [c for c in allowlist if str(c).startswith("Actuarial_")]

    print("✅ Feature audit plan configured")
    print("   plan:", plan_name)
    print("   allowlist count:", len(allowlist))
    print("   expected total:", expected_total)
    print("   actuarial in allowlist:", len(act_cols), act_cols)

    if len(allowlist) != expected_total:
        print("⚠️ Allowlist count differs from expected total. Check src/config.py")

    return cfg


In [9]:
TRAIN_RANDOM_SEED = 42

train_config = deepcopy(get_active_config("phase1"))

# Optional: override analysis horizon
# train_config["ANALYSIS_END_DATE"] = "2025-09-01"

train_config = enforce_feature_audit_plan(train_config)

# Force fresh dataset build and market data re-download
if "train_phase1_data" in globals():
    del train_phase1_data

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir="/content/tcn_tape_vectorized_version_clean/data_exports",
)


✅ Feature audit plan configured
   plan: exp6_feature_audit_20260221_v2
   allowlist count: 49
   expected total: 49
   actuarial in allowlist: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']
📊 Loading raw market data...
   ✅ Raw data shape: (55107, 7)
   ✅ Date range: 2003-09-02 00:00:00 → 2025-08-29 00:00:00

🔧 Computing multi-horizon log returns: [1, 5, 10, 21]
   ✅ Shape after returns: (54897, 11)

📈 Calculating 21-day rolling statistics

🧮 Computing technical indicators

📊 Computing dynamic covariance features

🎯 Adding regime awareness features
   ✅ Master DF shape: (54897, 47)
   ✅ Total features: 47

📊 Integrating fundamental features (if enabled)...
   ✅ Fundamental columns in dataset: 0 (enabled=False)

📊 Integrating macroeconomic features (if enabled)...
   ✅ Macro features added - 43 columns: ['EFFR_diff', 'EFFR_zscore', 'SOFR_level', 'SOFR_diff', 'FEDFUNDS_diff', 'FEDFUNDS_zscore', 'DGS10_level', 'DGS10_diff', 'DG


💾 Saving NORMALISED master dataframe to '/content/tcn_tape_vectorized_version_clean/data/master_features_NORMALIZED.csv'

💾 Saved preparation artifacts:
   raw OHLCV: /content/tcn_tape_vectorized_version_clean/data_exports/phase1_prep_20260225_093053_raw_ohlcv.csv
   full engineered: /content/tcn_tape_vectorized_version_clean/data_exports/phase1_prep_20260225_093053_feature_engineered_full.csv
   analysis-window engineered: /content/tcn_tape_vectorized_version_clean/data_exports/phase1_prep_20260225_093053_feature_engineered_analysis_window.csv
   normalized master: /content/tcn_tape_vectorized_version_clean/data_exports/phase1_prep_20260225_093053_feature_engineered_normalized.csv
   train normalized: /content/tcn_tape_vectorized_version_clean/data_exports/phase1_prep_20260225_093053_train_normalized.csv
   test normalized: /content/tcn_tape_vectorized_version_clean/data_exports/phase1_prep_20260225_093053_test_normalized.csv
   scalers: /content/tcn_tape_vectorized_version_clean/dat

In [10]:
print("Train shape:", train_phase1_data.train_df.shape)
print("Test shape:", train_phase1_data.test_df.shape)

cols = train_phase1_data.train_df.columns
print("Total columns:", len(cols))

# quick sanity for common redundant groups
dup_like = [c for c in cols if c.endswith("_raw") or c.endswith("_unscaled")]
print("Potential redundant raw/unscaled cols:", len(dup_like))
print(dup_like[:20])

used_now = list(dict.fromkeys(train_phase1_data.data_processor.get_feature_columns("phase1")))
act_now = [c for c in used_now if c.startswith("Actuarial_")]
print("Model feature count (phase1):", len(used_now))
print("Actuarial feature count:", len(act_now), act_now)


Train shape: (43867, 103)
Test shape: (11030, 103)
Total columns: 103
Potential redundant raw/unscaled cols: 0
[]
Model feature count (phase1): 49
Actuarial feature count: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']


In [11]:
[cols]

In [11]:
train_phase1_data.train_df[['Actuarial_Prob_60d', 'Actuarial_Prob_30d']].head()

,Actuarial_Prob_60d,Actuarial_Prob_30d
0,0.6,1.045455
1,0.6,1.045455
2,0.6,1.045455
3,0.6,1.045455
4,0.6,1.045455


In [12]:
used = set(train_phase1_data.data_processor.get_feature_columns("phase1"))
disabled = set(train_config["feature_params"]["feature_selection"]["disabled_features"])
act_used = sorted([c for c in used if c.startswith("Actuarial_")])

print("Used feature count:", len(used))
print("Actuarial used:", len(act_used), act_used)
print("Disabled that still in used:", sorted(disabled & used))  # should be []
print("VIX_zscore used?", "VIX_zscore" in used)


Used feature count: 49
Actuarial used: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']
Disabled that still in used: []
VIX_zscore used? True


In [13]:
base_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]
keep = [c for c in base_cols + list(used) if c in train_phase1_data.master_df.columns]

train_phase1_data.master_df = train_phase1_data.master_df[keep].copy()
train_phase1_data.train_df = train_phase1_data.train_df[keep].copy()
train_phase1_data.test_df  = train_phase1_data.test_df[keep].copy()

## 4) Training Overrides (Sharpe-Only Checkpoint Policy)

This policy keeps only Sharpe-threshold high-watermark checkpointing (`>= 0.5`) and disables rare/step/periodic/TAPE checkpoint routes.

In [14]:
# ============================================================================
# NEXT RUN OVERRIDES (throughput-first + stable policy improvement)
# ============================================================================
from copy import deepcopy

train_config = deepcopy(train_config)  # or deepcopy(config) if that's your active object

tp = train_config["training_params"]
ap = train_config["agent_params"]
ppo = ap["ppo_params"]
env = train_config["environment_params"]
fp = train_config.setdefault("feature_params", {})
fund_cfg = fp.setdefault("fundamental_features", {})
act_cfg = fp.setdefault("actuarial_params", {})

# Hard requirement for this run: no fundamentals, actuarial ON.
fund_cfg["enabled"] = False
act_cfg["enabled"] = True

# ----------------------------------------------------------------------------
# 1) Core run shape
# ----------------------------------------------------------------------------
tp["max_total_timesteps"] = 150_000
tp["timesteps_per_ppo_update"] = 1008  # fallback
tp["num_parallel_envs"] = 4  # vectorized rollout collection

tp["timesteps_per_ppo_update_schedule"] = [
    {"threshold": 0, "timesteps_per_update": 1008},   # 252/env × 4 envs
    {"threshold": 50_000, "timesteps_per_update": 1512},  # 378/env × 4 envs
]

tp["batch_size_ppo_schedule"] = [
    {"threshold": 0, "batch_size": 252},
    {"threshold": 50_000, "batch_size": 252},
]


# ----------------------------------------------------------------------------
# 2) A2: deeper temporal receptive field (1D TCN only; 2D variant skipped)
# ----------------------------------------------------------------------------
ap["tcn_filters"] = [64, 96, 128, 128, 128]
ap["tcn_kernel_size"] = 5
ap["tcn_dilations"] = [1, 2, 4, 8, 16]
ap["tcn_dropout"] = 0.15

# ----------------------------------------------------------------------------
# 3) PPO stability (allow learning while keeping KL controlled)
# ----------------------------------------------------------------------------
ppo["num_ppo_epochs"] = 2
ppo["policy_clip"] = 0.08
ppo["target_kl"] = 0.040
ppo["kl_stop_multiplier"] = 1.50
ppo["minibatches_before_kl_stop"] = 2
ppo["max_grad_norm"] = 0.30

ppo["actor_lr"] = 2.5e-5
ppo["critic_lr"] = 1.2e-4
ppo["entropy_coef"] = 0.0050

# Optional risk-aware actor auxiliaries (activate)
ppo["use_risk_aux_loss"] = True
ppo["risk_aux_return_feature_index"] = 0
ppo["risk_aux_cash_return"] = 0.0
ppo["risk_aux_sharpe_coef"] = 0.0
ppo["risk_aux_mvo_coef"] = 0.0
ppo["risk_aux_cvar_coef"] = 0.002
ppo["risk_aux_cvar_alpha"] = 0.05
ppo["risk_aux_mvo_cov_ridge"] = 1e-3
ppo["risk_aux_mvo_long_only"] = True
ppo["risk_aux_mvo_risky_budget"] = 0.95

tp["actor_lr_schedule"] = [
    {"threshold": 0, "lr": 2.5e-5},
    {"threshold": 30_000, "lr": 2.0e-5},
    {"threshold": 60_000, "lr": 1.5e-5},
]

# ----------------------------------------------------------------------------
# 4) RA-KL (disable in early stage; re-enable only after baseline improves)
# ----------------------------------------------------------------------------
tp["ra_kl_enabled"] = False
tp["ra_kl_target_ratio"] = 1.0
tp["ra_kl_ema_alpha"] = 0.25
tp["ra_kl_gain"] = 0.03
tp["ra_kl_deadband"] = 0.20
tp["ra_kl_max_change_fraction"] = 0.05
tp["ra_kl_min_target_kl"] = 0.016
tp["ra_kl_max_target_kl"] = 0.030

# ----------------------------------------------------------------------------
# 5) Dirichlet + concentration controls
# ----------------------------------------------------------------------------
ap["dirichlet_alpha_activation"] = "softplus"
ap["dirichlet_logit_temperature"] = 1.0  # Keep static temperature neutral when adaptive is on
ap["dirichlet_alpha_cap"] = 20.0
ap["dirichlet_epsilon"] = {"max": 0.2, "min": 0.02}

ap["dirichlet_adaptive_temperature_enabled"] = True
ap["dirichlet_adaptive_temperature_base"] = 0.9
ap["dirichlet_adaptive_temperature_slope"] = 0.6
ap["dirichlet_adaptive_temperature_min"] = 0.8
ap["dirichlet_adaptive_temperature_max"] = 2.5

# A3/A4: richer alpha head + optional cross-asset mixer
ap["fusion_cross_asset_mixer_enabled"] = True
ap["fusion_cross_asset_mixer_layers"] = 2
ap["fusion_cross_asset_mixer_expansion"] = 2.0
ap["fusion_cross_asset_mixer_dropout"] = 0.10
ap["fusion_alpha_head_hidden_dims"] = [128, 64]
ap["fusion_alpha_head_dropout"] = 0.05

env["concentration_penalty_scalar"] = 3.0
env["concentration_target_hhi"] = 0.12
env["top_weight_penalty_scalar"] = 2.0
env["action_realization_penalty_scalar"] = 0.5

# ----------------------------------------------------------------------------
# 6) Turnover + execution smoothing
# ----------------------------------------------------------------------------
env["target_turnover"] = 0.35
env["turnover_penalty_scalar"] = 0.05
env["transaction_cost_pct"] = 0.001

tp["action_execution_beta_curriculum"] = {
    0: 0.10,
    30_000: 0.20,
    60_000: 0.35,
}
tp["evaluation_action_execution_beta"] = 0.25

tp["turnover_penalty_curriculum"] = {
    0: 0.05,
    10_000: 0.10,
    25_000: 0.15,
    40_000: 0.20,
}
tp["evaluation_turnover_penalty_scalar"] = 0.2

# ----------------------------------------------------------------------------
# 7) Episode horizon curriculum (keep cap late)
# ----------------------------------------------------------------------------
tp["use_episode_length_curriculum"] = True
tp["episode_length_curriculum_schedule"] = [
    {"threshold": 0, "limit": 252},
    {"threshold": 10_000, "limit": 504},
    {"threshold": 25_000, "limit": 756},
    {"threshold": 90_000, "limit": 1008},
]

# ----------------------------------------------------------------------------
# 8) Logging + checkpoints (reduce validation overhead)
# ----------------------------------------------------------------------------
tp["log_step_diagnostics"] = True
tp["update_log_interval"] = 10
tp["alpha_diversity_log_interval"] = 2
tp["alpha_diversity_warning_after_updates"] = 120
tp["alpha_diversity_warning_std_threshold"] = 0.25

tp["deterministic_validation_checkpointing_enabled"] = False
tp["deterministic_validation_eval_every_episodes"] = 5
tp["deterministic_validation_mode"] = "mean"
tp["deterministic_validation_episode_length_limit"] = None
tp["deterministic_validation_episode_length_limit_curriculum"] = [
    {"threshold": 0, "limit": 252},
    {"threshold": 30_000, "limit": 504},
    {"threshold": 70_000, "limit": 756},
    {"threshold": 100_000, "limit": 1008},
]
tp["deterministic_validation_sharpe_min"] = 0.5
tp["deterministic_validation_sharpe_min_delta"] = 0.005
tp["deterministic_validation_seed_offset"] = 10_000
tp["deterministic_validation_log_alpha_stats"] = True
tp["deterministic_validation_checkpointing_only"] = False

tp["high_watermark_checkpoint_enabled"] = True
tp["high_watermark_sharpe_threshold"] = 0.5
tp["high_watermark_max_drawdown_abs_threshold"] = 0.25
tp["high_watermark_skip_on_deterministic_validation_trigger"] = True
tp["step_sharpe_checkpoint_enabled"] = False
tp["periodic_checkpoint_every_steps"] = 0
tp["rare_checkpoint_params"] = {"enable": False}
tp["tape_checkpoint_threshold"] = 999.0


# ----------------------------------------------------------------------------
# 9) Long-horizon upgrades (all enabled)
# ----------------------------------------------------------------------------
ap["recurrent_memory_enabled"] = True
ap["recurrent_memory_units"] = 64
ap["recurrent_memory_dropout"] = 0.10

ap["regime_conditioning_enabled"] = True
ap["regime_conditioning_hidden_dim"] = 32
ap["regime_conditioning_dropout"] = 0.0
ap["state_augmentation_enabled"] = True

ap["distributional_critic_enabled"] = True
ap["distributional_num_quantiles"] = 17

ppo["popart_enabled"] = True
ppo["popart_min_std"] = 1e-3

ppo["multi_horizon_reward_enabled"] = True
ppo["multi_horizon_reward_coef"] = 0.20
ppo["multi_horizon_reward_horizons"] = [21, 63, 126, 252]
ppo["multi_horizon_reward_weights"] = [0.15, 0.25, 0.30, 0.30]

ppo["risk_aux_cvar_adaptive_enabled"] = True
ppo["risk_aux_cvar_target"] = 0.015
ppo["risk_aux_cvar_adapt_lr"] = 0.05
ppo["risk_aux_cvar_min_coef"] = 0.0
ppo["risk_aux_cvar_max_coef"] = 0.06

tp["episode_length_curriculum_smooth_enabled"] = True
tp["episode_length_curriculum_overlap_steps"] = 10_000

tp["ppo_gamma_schedule"] = [
    {"threshold": 0, "gamma": 0.985},
    {"threshold": 50_000, "gamma": 0.992},
    {"threshold": 100_000, "gamma": 0.997},
]
tp["ppo_gae_lambda_schedule"] = [
    {"threshold": 0, "gae_lambda": 0.90},
    {"threshold": 50_000, "gae_lambda": 0.94},
    {"threshold": 100_000, "gae_lambda": 0.97},
]

tp["deterministic_validation_multi_horizon_enabled"] = False
tp["deterministic_validation_multi_horizon_limits"] = [252, 504, 756, 1008]
tp["deterministic_validation_multi_horizon_weights"] = [0.35, 0.30, 0.20, 0.15]
tp["deterministic_validation_multi_horizon_dd_penalty_coef"] = 0.25
tp["deterministic_validation_stochastic_sanity_enabled"] = False
tp["deterministic_validation_stochastic_sanity_runs"] = 3
tp["deterministic_validation_stochastic_sanity_episode_length_limit"] = 252
tp["deterministic_validation_stochastic_sanity_min_mean_sharpe"] = 0.0
tp["deterministic_validation_stochastic_sanity_max_sharpe_std"] = 1.5

print("✅ Applied next-run override (throughput-first + stable policy improvement)")
print("num_ppo_epochs:", ppo["num_ppo_epochs"])
print("target_kl:", ppo["target_kl"], "| kl_stop_multiplier:", ppo["kl_stop_multiplier"])
print("risk_aux:", {k: ppo[k] for k in ["use_risk_aux_loss", "risk_aux_sharpe_coef", "risk_aux_mvo_coef", "risk_aux_cvar_coef", "risk_aux_cvar_alpha", "risk_aux_return_feature_index", "risk_aux_mvo_risky_budget"]})
print("RA-KL:", {k: tp[k] for k in [
    "ra_kl_enabled", "ra_kl_gain", "ra_kl_deadband",
    "ra_kl_max_change_fraction", "ra_kl_min_target_kl", "ra_kl_max_target_kl"
]})
print("num_parallel_envs:", tp["num_parallel_envs"])
print("action_execution_beta_curriculum:", tp["action_execution_beta_curriculum"])
print("turnover_penalty_curriculum:", tp["turnover_penalty_curriculum"])
print("det-val every episodes:", tp["deterministic_validation_eval_every_episodes"], "| horizon:", tp["deterministic_validation_episode_length_limit"])
print("det-val horizon curriculum:", tp["deterministic_validation_episode_length_limit_curriculum"])
print("high-watermark enabled:", tp["high_watermark_checkpoint_enabled"])
print("high-watermark thresholds: sharpe>=", tp["high_watermark_sharpe_threshold"], "| mdd<=", tp["high_watermark_max_drawdown_abs_threshold"])
print("high-watermark skip on det-validation trigger:", tp["high_watermark_skip_on_deterministic_validation_trigger"])
print("deterministic_validation_checkpointing_only:", tp["deterministic_validation_checkpointing_only"])
print("concentration:", env["concentration_penalty_scalar"], env["concentration_target_hhi"], env["top_weight_penalty_scalar"])

print("TCN stack:", ap["tcn_filters"], "| dilations:", ap["tcn_dilations"], "| dropout:", ap["tcn_dropout"])
print("Fusion mixer:", {k: ap[k] for k in ["fusion_cross_asset_mixer_enabled", "fusion_cross_asset_mixer_layers", "fusion_cross_asset_mixer_expansion", "fusion_cross_asset_mixer_dropout"]})
print("Fusion alpha head:", ap["fusion_alpha_head_hidden_dims"], "| dropout:", ap["fusion_alpha_head_dropout"])
print("fundamental_features.enabled:", bool(fund_cfg.get("enabled", False)))
print("actuarial_params.enabled:", bool(act_cfg.get("enabled", False)))

print("recurrent memory:", {k: ap[k] for k in ["recurrent_memory_enabled", "recurrent_memory_units", "recurrent_memory_dropout"]})
print("regime conditioning:", {k: ap[k] for k in ["regime_conditioning_enabled", "regime_conditioning_hidden_dim", "regime_conditioning_dropout", "state_augmentation_enabled"]})
print("distributional critic:", {k: ap[k] for k in ["distributional_critic_enabled", "distributional_num_quantiles"]})
print("PopArt + reward decomposition:", {k: ppo[k] for k in ["popart_enabled", "popart_min_std", "multi_horizon_reward_enabled", "multi_horizon_reward_coef", "multi_horizon_reward_horizons", "multi_horizon_reward_weights"]})
print("ppo gamma/gae schedules:", tp["ppo_gamma_schedule"], tp["ppo_gae_lambda_schedule"])
print("episode horizon smooth ramp:", tp["episode_length_curriculum_smooth_enabled"], "overlap:", tp["episode_length_curriculum_overlap_steps"])
print("det-val multi-horizon+sanity:", {
    "enabled": tp["deterministic_validation_multi_horizon_enabled"],
    "limits": tp["deterministic_validation_multi_horizon_limits"],
    "weights": tp["deterministic_validation_multi_horizon_weights"],
    "dd_penalty": tp["deterministic_validation_multi_horizon_dd_penalty_coef"],
    "sanity_enabled": tp["deterministic_validation_stochastic_sanity_enabled"],
})


✅ Applied next-run override (throughput-first + stable policy improvement)
num_ppo_epochs: 2
target_kl: 0.04 | kl_stop_multiplier: 1.5
risk_aux: {'use_risk_aux_loss': True, 'risk_aux_sharpe_coef': 0.0, 'risk_aux_mvo_coef': 0.0, 'risk_aux_cvar_coef': 0.005, 'risk_aux_cvar_alpha': 0.05, 'risk_aux_return_feature_index': 0, 'risk_aux_mvo_risky_budget': 0.95}
RA-KL: {'ra_kl_enabled': False, 'ra_kl_gain': 0.03, 'ra_kl_deadband': 0.2, 'ra_kl_max_change_fraction': 0.05, 'ra_kl_min_target_kl': 0.016, 'ra_kl_max_target_kl': 0.03}
num_parallel_envs: 4
action_execution_beta_curriculum: {0: 0.05, 30000: 0.15, 60000: 0.25}
turnover_penalty_curriculum: {0: 0.05, 10000: 0.1, 25000: 0.15, 40000: 0.2}
det-val every episodes: 5 | horizon: None
det-val horizon curriculum: [{'threshold': 0, 'limit': 252}, {'threshold': 30000, 'limit': 504}, {'threshold': 70000, 'limit': 756}, {'threshold': 100000, 'limit': 1008}]
high-watermark enabled: True
high-watermark thresholds: sharpe>= 0.5 | mdd<= 0.25
high-waterma

In [15]:
# Optional experimental override (OFF by default)
# Keep this OFF for the aligned default pipeline.
EXPERIMENT_DISABLE_KL_GUARDS = False

if EXPERIMENT_DISABLE_KL_GUARDS:
    tp = train_config["training_params"]
    ppo = train_config["agent_params"]["ppo_params"]

    # Disable RA-KL controller (otherwise it keeps adjusting target_kl)
    tp["ra_kl_enabled"] = False

    # Disable KL early-stop gate in PPOAgentTF
    ppo["target_kl"] = 0.0

    # Optional (irrelevant once target_kl=0, but explicit)
    ppo["kl_stop_multiplier"] = 999.0
    ppo["minibatches_before_kl_stop"] = 9999
    print("⚠️ EXPERIMENT_DISABLE_KL_GUARDS=True (non-default experimental mode)")
else:
    print("ℹ️ EXPERIMENT_DISABLE_KL_GUARDS=False (keeping RA-KL + KL safeguards)")



ℹ️ EXPERIMENT_DISABLE_KL_GUARDS=False (keeping RA-KL + KL safeguards)


## 5) Run Training

In [16]:
RUN_TRAINING = True

if RUN_TRAINING:
    tp = train_config["training_params"]
    print("🚀 Starting training")
    print("Architecture:", train_config["agent_params"].get("actor_critic_type"))
    print("max_total_timesteps:", tp["max_total_timesteps"])
    print("num_parallel_envs:", tp.get("num_parallel_envs", 1))

    actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith("Actuarial_")]
    if not actuarial_cols:
        raise RuntimeError("Actuarial features missing in train_phase1_data.master_df")
    actuarial_non_null = {c: int(train_phase1_data.master_df[c].notna().sum()) for c in actuarial_cols}
    if any(v == 0 for v in actuarial_non_null.values()):
        raise RuntimeError(f"Actuarial features present but empty: {actuarial_non_null}")

    fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith("Fundamental_")]
    if fundamental_cols:
        raise RuntimeError(f"Fundamental columns still present (expected removed): {fundamental_cols}")

    print("✅ Actuarial feature check passed:", actuarial_non_null)
    print("✅ Fundamental feature check passed: none present")

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config["agent_params"].get("actor_critic_type"),
        timesteps_per_update=tp.get("timesteps_per_ppo_update", 384),
        max_total_timesteps=tp["max_total_timesteps"],
    )

    print("✅ Training complete")
    print("checkpoint_prefix:", train_experiment6.checkpoint_path)
else:
    print("ℹ️ RUN_TRAINING=False")



🚀 Starting training
Architecture: TCN_FUSION
max_total_timesteps: 150000
num_parallel_envs: 4
✅ Actuarial feature check passed: {'Actuarial_Prob_60d': 54897, 'Actuarial_Expected_Recovery': 54897, 'Actuarial_Prob_30d': 54897, 'Actuarial_Reserve_Severity': 54897}
✅ Fundamental feature check passed: none present

EXPERIMENT 6: TCN_FUSION Enhanced + TAPE Three-Component
Architecture: TCN + Fusion
Results root: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results
Working dir: /content/tcn_tape_vectorized_version_clean
Covariance Features: Yes
🎯 REWARD SYSTEM: TAPE (Three-Component v3)
   Profile: BalancedGrowth
   Daily: Base + DSR/PBRS + Turnover_Proximity
   Terminal: mode=signed | baseline=0.20 | scalar=10.0 (clipped ±10.0)
   Gate A: enabled (Sharpe ≤ 0.00 or MDD ≥ 25.0% -> force non-positive terminal bonus)
   Neutral Band: enabled (±0.020 around baseline)
   🔄 Profile Manager: disabled (static profile only)
🎲 Experiment Seed: 6042 (Base: 42, Offset: 6000)
✅ Features: Enhanced


📚 EPISODE HORIZON UPDATE at 2,016 steps:
   Episode horizon: 303 steps


      🧪 Deterministic validation: Sharpe=0.659 | Return=+47.46% | DD=13.92%
         Multi-horizon: score=0.790 | details=252:1.207/8.9%, 504:0.487/13.9%, 756:0.763/13.9%, 1008:0.659/13.9%
         Stochastic sanity: mean_sharpe=0.914 | std=0.460 | runs=3
      💾 Deterministic-validation checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00005_shp0p659_actor.weights.h5 (val_sharpe=0.659, score=0.790)
      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00008_shp2p173_actor.weights.h5 (Sharpe=2.173, MDD=4.17%)

📚 EPISODE HORIZON UPDATE at 3,024 steps:
   Episode horizon: 328 steps


      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00009_shp1p232_actor.weights.h5 (Sharpe=1.232, MDD=3.82%)
      🧪 Deterministic validation: Sharpe=0.661 | Return=+47.47% | DD=13.80%
         Multi-horizon: score=0.798 | details=252:1.217/8.9%, 504:0.496/13.8%, 756:0.770/13.8%, 1008:0.661/13.8%
         Stochastic sanity: mean_sharpe=0.368 | std=0.653 | runs=3
      💾 Deterministic-validation checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00010_shp0p661_actor.weights.h5 (val_sharpe=0.661, score=0.798)
      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00012_shp0p753_actor.weights.h5 (Sharpe=0.753, MDD=14.24%)

📚 EPISODE HORIZON UPDATE at 4,032 steps:
   Episode horizon: 354 steps


      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00013_shp1p130_actor.weights.h5 (Sharpe=1.130, MDD=9.44%)
      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00014_shp1p224_actor.weights.h5 (Sharpe=1.224, MDD=6.70%)
      🧪 Deterministic validation: Sharpe=0.660 | Return=+47.42% | DD=13.79%
         Multi-horizon: score=0.799 | details=252:1.218/8.8%, 504:0.497/13.8%, 756:0.773/13.8%, 1008:0.660/13.8%
      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00016_shp0p764_actor.weights.h5 (Sharpe=0.764, MDD=9.31%)

📚 EPISODE HORIZON UPDATE at 5,040 steps:
   Episode horizon: 379 steps


...


📚 TURNOVER CURRICULUM UPDATE at 10,080 steps:
   Turnover penalty scalar: 0.1

📚 EPISODE HORIZON UPDATE at 10,080 steps:
   Episode horizon: 504 steps


🔄 Update 10/116 | Step 10,080/150,000 | Episode 24 | Time: 7423.4s
   📊 Metrics: Return=+23.27% | Sharpe=0.795 | DD=10.54% | Turnover=3.06%
   🎚️ Intra-Step TAPE: potential=0.7556 | delta_reward=-0.0000
   🎯 Profile: BalancedGrowth
   🧠 Training: actor_loss=0.2875 | critic_loss=0.3155 | mean_adv=0.0000
   🧮 Loss Detail: critic_scaled=0.1578 | risk_aux_total=0.1768 | sharpe_proxy=0.0000 | sharpe_loss=0.0000 | mvo_loss=0.0000 | cvar_proxy=2.2105 | cvar_loss=0.1768 | cvar_coef=0.0800
   ⚙️ Optimizer: actor_lr=0.000025 | critic_lr=0.000120 | target_kl=0.0400 | rollout=1008 | batch_size=252 | gamma=0.9850 | gae_lambda=0.9000
   🔬 Alpha Diversity: mean=1.04 | std=0.24 | range=[0.62, 1.45]
      🧪 Deterministic validation: Sharpe=0.665 | Return=+47.93% | DD=13.94%
         Multi-horizon: score=0.800 | details=252:1.213/8.8%, 504:0.497/13.9%, 756:0.786/13.9%, 1008:0.665/13.9%


      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00029_shp1p717_actor.weights.h5 (Sharpe=1.717, MDD=4.23%)
      🧪 Deterministic validation: Sharpe=0.667 | Return=+48.26% | DD=13.97%
         Multi-horizon: score=0.804 | details=252:1.218/8.8%, 504:0.500/14.0%, 756:0.793/14.0%, 1008:0.667/14.0%
         Stochastic sanity: mean_sharpe=0.531 | std=0.476 | runs=3
      💾 Deterministic-validation checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00030_shp0p667_actor.weights.h5 (val_sharpe=0.667, score=0.804)
      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00031_shp0p597_actor.weights.h5 (Sharpe=0.597, MDD=14.45%)
      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_wate

      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00033_shp1p148_actor.weights.h5 (Sharpe=1.148, MDD=6.75%)


: 

## 6) Inspect Latest Training Logs

In [18]:
TRAIN_RESULTS_ROOT = Path("/content/tcn_tape_vectorized_version_clean/tcn_fusion_results")
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / "logs"

episodes_files = sorted(TRAIN_LOGS_DIR.glob("*episodes*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f"No episodes CSV found in {TRAIN_LOGS_DIR}")
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print("Episodes file:", train_episodes_path)
    print("Rows:", len(train_episodes_df))
    display(train_episodes_df.tail(20))

In [20]:
#train_episodes_df.columns

## 7) Export Results Folder (Optional)
Creates a zip for download from Colab VM.

In [21]:
from pathlib import Path
import subprocess

EXPORT_RESULTS_ZIP = True
EXPORT_PATH = Path("/content/tcn_tape_vectorized_run1.zip")
ROOT = Path("/content/tcn_tape_vectorized_version_clean")

if EXPORT_RESULTS_ZIP:
    # Core items
    include_paths = [
        ROOT / "tcn_fusion_results",
        ROOT / "data" / "phase1_preparation_artifacts",
        ROOT / "data" / "master_features_NORMALIZED.csv",
        ROOT / "data_exports",  # include all prep exports like phase1_prep_* artifacts
    ]

    # Also include latest phase1_prep_* files (explicitly, if present)
    data_exports_dir = ROOT / "data_exports"
    if data_exports_dir.exists():
        latest_prep_files = sorted(
            data_exports_dir.glob("phase1_prep_*"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        include_paths.extend(latest_prep_files)

    # De-dup + existence check
    seen = set()
    existing = []
    for p in include_paths:
        p = p.resolve()
        if p.exists() and p not in seen:
            seen.add(p)
            existing.append(p)

    if not existing:
        print("⚠️ Nothing to export.")
    else:
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()

        rel_items = [str(p.relative_to(ROOT)) for p in existing if str(p).startswith(str(ROOT))]
        if not rel_items:
            print("⚠️ No export items are under ROOT.")
        else:
            cmd = f"cd {ROOT} && zip -qr {EXPORT_PATH} " + " ".join(f'"{x}"' for x in rel_items)
            subprocess.run(cmd, shell=True, check=True)

            print(f"✅ Created: {EXPORT_PATH}")
            print("Included:")
            for p in rel_items:
                print(" -", p)
else:
    print("ℹ️ EXPORT_RESULTS_ZIP=False")

In [23]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/tcn_tape_vectorized_run1.zip /content/drive/MyDrive/
print("✅ Copied to Drive: /content/drive/MyDrive/tcn_tape_vectorized_run1.zip")